# Experimento Colaborativo #07: Utilización de los meses de pandemia
Modalidad Gerencial, Grupo A. Facundo Londero.

Es el workflow **z610** de la cátedra tal cual, corrido para 4 variantes del primer mes de training y 10 semillas. Validation 202107, entrenamiento final hasta 202107 y future 202109 (Kaggle). En cada corrida se vuelve a hacer el grid search completo.

| Variante | Primer mes de training | Meses de training |
|---|---|---|
| V0 línea base (todos los meses) | 202005 | 14 |
| V1 sin ASPO | 202011 | 8 |
| V2 solo 2021 | 202101 | 6 |
| V3 últimos cuatro meses | 202103 | 4 |

Una diferencia con el notebook, igual en las cuatro variantes: el modelo final usa `modifyList(fijos, mejores)` en vez de `c(fijos, mejores)`. Con `c()` quedan num_leaves y min_data_in_leaf duplicados, LightGBM se queda con el primero (64 y 128) e ignora lo que eligió el grid search.

Por corrida se suben a Kaggle 6 cortes, los del notebook de a 100 (800, 900, 1000, 1100, 1200 y 1300 envíos). La ganancia de cada corrida es el promedio Public de esos 6 cortes.

**Cómo correrlo:** el entorno de ejecución tiene que ser **R**. Con `modo <- "test"` es una prueba de 2 minutos; con `"full"` son las 40 corridas (en Colab conviene de a una semilla). Con `subir_kaggle <- TRUE` sube a Kaggle los 6 archivos de cada corrida apenas termina; si una subida falla, queda anotada en `pendientes_kaggle.txt` y se reintenta al volver a correr la celda de las corridas. La primera vez instala LightGBM, que tarda unos 7 minutos. Para que los resultados queden en el Drive, correr antes las dos celdas de arranque del z610 con el entorno en Python.

In [ ]:
modo <- "test"           # "test" o "full"
subir_kaggle <- FALSE    # TRUE sube a Kaggle los 6 archivos de cada corrida apenas termina (necesita el kaggle.json)

# semillas: las mismas 10 en las 4 variantes, porque la comparacion es pareada por semilla
semillas <- c(553897, 761897, 815197, 906187, 919531,
              150001, 222247, 199999, 777781, 999983)

# variantes: lo unico que cambia es el primer mes de training
variantes <- list(
  list(nombre = "V0_todo",     desde = 202005,
       descripcion = "linea base de la catedra: todos los meses, incluye el ASPO (202005-202106)"),
  list(nombre = "V1_sinASPO",  desde = 202011,
       descripcion = "sin los meses de ASPO: desde noviembre 2020, cuando el AMBA pasa a DISPO (202011-202106)"),
  list(nombre = "V2_2021",     desde = 202101,
       descripcion = "solo 2021 (202101-202106)"),
  list(nombre = "V3_ultimos4", desde = 202103,
       descripcion = "solo los ultimos 4 meses de training (202103-202106)")
)

MES_VALIDATE <- 202107   # validation del grid search
MES_HASTA    <- 202106   # ultimo mes de training
MES_FUTURE   <- 202109   # mes que se predice (Kaggle)
CORTES       <- seq(800, 1300, by = 100)   # los cortes del notebook, de a 100
COMPETENCIA  <- "labo-1-virtual-2026-manager"

# Kaggle: el kaggle.json lo copia la celda de arranque del z610; si no, lo busco en el Drive o en /content
# (subiendolo con el panel de archivos de Colab). Hace falta para subir desde aca y para bajar los puntajes.
preparar_kaggle <- function() {
  for (origen in c("/content/buckets/b1/kaggle/kaggle.json", "/content/kaggle.json")) {
    if (!file.exists("~/.kaggle/kaggle.json") && file.exists(origen)) {
      dir.create("~/.kaggle", showWarnings = FALSE)
      file.copy(origen, "~/.kaggle/kaggle.json")
      Sys.chmod("~/.kaggle/kaggle.json", "600")
    }
  }
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json: subilo a /content/kaggle.json con el panel de archivos de Colab")
  invisible(system("python3 -m pip install -q --disable-pip-version-check kaggle==2.2.4"))
}

# sube un archivo a Kaggle con el mensaje "exp07 <variante> s<semilla> envios=<corte>". Si la subida falla (se agoto el
# cupo de 100 envios por dia, se corto la red) lo anota en pendientes_kaggle.txt, y reintentar_pendientes() lo vuelve
# a intentar al correr de nuevo la celda de las corridas.
subir_a_kaggle <- function(archivo) {
  mensaje <- sub("^KA07_(.+)_s([0-9]+)_([0-9]+)\\.csv$", "exp07 \\1 s\\2 envios=\\3", basename(archivo))
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", COMPETENCIA, "-f", shQuote(archivo),
                                         "-m", shQuote(mensaje)), stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) Sys.sleep(30) else cat(archivo, "\n", sep = "", file = "pendientes_kaggle.txt", append = TRUE)
  ok
}

reintentar_pendientes <- function() {
  if (!subir_kaggle || !file.exists("pendientes_kaggle.txt")) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines("pendientes_kaggle.txt")), ""))
  file.remove("pendientes_kaggle.txt")
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (i in seq_along(pendientes)) {
    if (!subir_a_kaggle(pendientes[i])) {   # si falla uno, seguro no hay cupo: los demas quedan para la proxima
      resto <- pendientes[-seq_len(i)]
      if (length(resto)) cat(paste0(resto, "\n"), sep = "", file = "pendientes_kaggle.txt", append = TRUE)
      break
    }
  }
}
if (subir_kaggle) preparar_kaggle()

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

## Preprocesamiento
Idéntico al z610: Catastrophe Analysis de 202006, FE intra-mes y lags y deltas de orden 1 y 2.

In [ ]:
catastrophe_analysis <- function(dataset) {
  # el notebook asigna NA a las variables que en 202006 estan todas en cero
  dataset[ foto_mes==202006, internet:=NA]
  dataset[ foto_mes==202006, mrentabilidad:=NA]
  dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
  dataset[ foto_mes==202006, mcomisiones:=NA]
  dataset[ foto_mes==202006, mactivos_margen:=NA]
  dataset[ foto_mes==202006, mpasivos_margen:=NA]
  dataset[ foto_mes==202006, mcuentas_saldo:=NA]
  dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
  dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
  dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
  dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
  dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
  invisible(dataset)
}

fe_intra_manual <- function(dataset) {
  atributos_presentes <- function(patributos) {
    atributos <- unique(patributos)
    comun <- intersect(atributos, colnames(dataset))
    return(length(atributos) == length(comun))
  }
  if (atributos_presentes(c("foto_mes")))
    dataset[, kmes := foto_mes %% 100]
  if (atributos_presentes(c("mpayroll", "cliente_edad")))
    dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]
  invisible(dataset)
}

fe_historico <- function(dataset) {
  # el notebook confia en que, dentro de cada cliente, las filas vienen en orden cronologico; lo verifico
  stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

  cols_lagueables <- copy(setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
  ))
  dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
          by = numero_de_cliente, .SDcols = cols_lagueables]
  dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
          by = numero_de_cliente, .SDcols = cols_lagueables]
  for (vcol in cols_lagueables) {
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
  }
  invisible(dataset)
}

preprocesar <- function(archivo_dataset) {
  dataset <- fread(archivo_dataset)
  catastrophe_analysis(dataset)
  fe_intra_manual(dataset)
  fe_historico(dataset)
  dataset
}

# meses de training: desde `desde` hasta MES_HASTA, en orden descendente como los escribe el notebook
meses_training <- function(desde, hasta = MES_HASTA) {
  todos <- c(202005, 202006, 202007, 202008, 202009, 202010, 202011, 202012,
             202101, 202102, 202103, 202104, 202105, 202106, 202107)
  rev(todos[todos >= desde & todos <= hasta])
}

## Modelado y producción para una variante y una semilla
Training strategy, grid search de 20 combinaciones, modelo final, predicción de 202109 y los 6 archivos para Kaggle.

In [ ]:
correr_experimento <- function(dataset, variante, semilla, carpeta,
                               grid = CJ(num_leaves = c(64, 128, 256, 512),
                                         min_data_in_leaf = c(64, 256, 512, 1024, 2048)),
                               num_iterations = 2048, early_stopping_rounds = 200) {

  t0 <- Sys.time()
  dir.create(carpeta, recursive = TRUE, showWarnings = FALSE)

  PARAM <- list()
  PARAM$problema <- "07 - Utilizacion de los meses de pandemia (Gerencial, Grupo A)"
  PARAM$semilla_primigenia <- semilla
  PARAM$variante <- variante$nombre
  PARAM$variante_descripcion <- variante$descripcion
  PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

  # --- 6.3.2.1 Training Strategy ---
  PARAM$trainingstrategy$validate <- c(MES_VALIDATE)
  PARAM$trainingstrategy$training <- meses_training(variante$desde)
  PARAM$trainingstrategy$training_pct <- 1.0
  PARAM$trainingstrategy$positivos <- c("BAJA+1", "BAJA+2")

  dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0)]

  # el dataset se reutiliza entre corridas: excluyo explicitamente las columnas auxiliares
  campos_buenos <- copy(setdiff(
    colnames(dataset),
    c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
  ))

  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]

  dataset[, fold_train := foto_mes %in% PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct)]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_train == TRUE, clase01],
    free_raw_data = TRUE
  )
  dvalidate <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
    free_raw_data = TRUE
  )

  PARAM$out$n_train <- dataset[fold_train == TRUE, .N]
  PARAM$out$pos_train <- dataset[fold_train == TRUE, sum(clase01)]
  PARAM$out$baja2_train <- dataset[fold_train == TRUE, sum(clase_ternaria == "BAJA+2")]

  # --- 6.3.2.2 Hyperparameter Tuning (grid search, identico al notebook) ---
  PARAM$lgbm$param_fijos <- list(
    objective = "binary",
    metric = "auc",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    force_row_wise = TRUE,
    seed = PARAM$semilla_primigenia,
    max_bin = 31,
    learning_rate = 0.03,
    feature_fraction = 0.5,
    num_iterations = num_iterations,
    early_stopping_rounds = early_stopping_rounds,
    num_leaves = 64,
    min_data_in_leaf = 128
  )

  Estimar_AUC_lightgbm <- function(x) {
    param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
    modelo_train <- lgb.train(
      data = dtrain,
      valids = list(valid = dvalidate),
      eval = "auc",
      param = param_completo,
      verbose = -100
    )
    AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
    niter <- modelo_train$best_iter
    message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, " s", semilla, "  ",
            toString(x), " niter ", niter, " AUC ", AUC)
    rm(modelo_train)
    gc(full = TRUE, verbose = FALSE)
    return(list(AUC, niter))
  }

  tb_nueva <- copy(grid)
  tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(as.list(.SD)),
           by = 1:nrow(tb_nueva)]
  tb_nueva[, `:=`(variante = variante$nombre, semilla = semilla)]
  fwrite(tb_nueva, file = file.path(carpeta, "tb_grid_search.txt"), sep = "\t")

  setorder(tb_nueva, -AUC)
  PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
  PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_nueva[1, list(num_leaves, min_data_in_leaf, num_iterations)])

  # --- 6.3.3 Produccion: Final Training ---
  PARAM$trainingstrategy$final_train <- c(MES_VALIDATE, PARAM$trainingstrategy$training)
  dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train]

  dfinal_train <- lgb.Dataset(
    data = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_final_train == TRUE, clase01],
    free_raw_data = TRUE
  )
  PARAM$out$n_final_train <- dataset[fold_final_train == TRUE, .N]

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL

  # el notebook hace c(fijos, mejores); modifyList pisa num_leaves y min_data_in_leaf en vez de duplicarlos
  param_final <- modifyList(fijos, PARAM$out$lgbm$mejores_hiperparametros)
  PARAM$out$param_final <- param_final

  final_model <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
  lgb.save(final_model, file.path(carpeta, "modelo.txt"))

  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite(tb_importancia, file = file.path(carpeta, "impo.txt"), sep = "\t")

  # --- Scoring del futuro (202109) ---
  PARAM$trainingstrategy$future <- c(MES_FUTURE)
  dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
  prediccion <- predict(final_model, data.matrix(dfuture[, campos_buenos, with = FALSE]))

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  fwrite(tb_prediccion, file = file.path(carpeta, "prediccion.txt"), sep = "\t")

  # --- Archivos para Kaggle ---
  PARAM$kaggle$competencia <- COMPETENCIA
  PARAM$kaggle$cortes <- CORTES
  setorder(tb_prediccion, -prob)
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  for (envios in PARAM$kaggle$cortes) {
    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]
    archivo_kaggle <- file.path(carpeta, "kaggle",
                                paste0("KA07_", variante$nombre, "_s", semilla, "_", envios, ".csv"))
    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

    if (subir_kaggle) subir_a_kaggle(archivo_kaggle)
  }

  PARAM$out$minutos <- round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 2)
  write_yaml(PARAM, file.path(carpeta, "PARAM.yml"))

  rm(dtrain, dvalidate, dfinal_train, final_model)
  gc(full = TRUE, verbose = FALSE)

  # una fila de resumen por corrida
  data.table(
    variante = variante$nombre, desde = variante$desde, semilla = semilla,
    n_train = PARAM$out$n_train, pos_train = PARAM$out$pos_train, baja2_train = PARAM$out$baja2_train,
    n_final_train = PARAM$out$n_final_train,
    AUC_valid = PARAM$out$lgbm$AUC,
    num_leaves = PARAM$out$lgbm$mejores_hiperparametros$num_leaves,
    min_data_in_leaf = PARAM$out$lgbm$mejores_hiperparametros$min_data_in_leaf,
    num_iterations = PARAM$out$lgbm$mejores_hiperparametros$num_iterations,
    minutos = PARAM$out$minutos
  )
}

## Corridas
Todas las variantes por todas las semillas, con la semilla afuera y la variante adentro. Saltea las corridas que ya tienen su PARAM.yml y, antes de empezar, reintenta las subidas a Kaggle que hayan quedado pendientes.

In [ ]:
# carpeta de trabajo: en el Drive si se corrieron las celdas de la catedra, si no en el disco de Colab
base <- if (dir.exists("/content/buckets/b1")) "/content/buckets/b1/exp/WF07_grupoA" else
        if (dir.exists("/content")) "/content/exp/WF07_grupoA" else file.path(getwd(), "WF07_grupoA")
dir.create(base, recursive = TRUE, showWarnings = FALSE)
setwd(base)

# dataset: el que baja la celda de la catedra o, si no esta, lo descargo
archivo_dataset <- "/content/datasets/gerencial_competencia_2026.csv.gz"
if (!file.exists(archivo_dataset)) {
  archivo_dataset <- file.path(base, "gerencial_competencia_2026.csv.gz")
  if (!file.exists(archivo_dataset))
    download.file("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/gerencial_competencia_2026.csv.gz",
                  archivo_dataset, mode = "wb")
}

if (modo == "test") {
  carpeta_runs <- "runs_test"
  archivo_resumen <- "resultados_test.csv"
  variantes <- variantes[c(1, length(variantes))]
  semillas <- semillas[1]
  grid <- CJ(num_leaves = c(64, 512), min_data_in_leaf = c(128))
  num_iterations <- 60
  early_stopping_rounds <- 20
} else {
  carpeta_runs <- "runs"
  archivo_resumen <- "resultados_grid.csv"
  grid <- CJ(num_leaves = c(64, 128, 256, 512),
             min_data_in_leaf = c(64, 256, 512, 1024, 2048))
  num_iterations <- 2048
  early_stopping_rounds <- 200
}

message(format(Sys.time(), "%a %b %d %X %Y"), "  modo=", modo, "  carpeta ", base)
t0 <- Sys.time()

reintentar_pendientes()   # subidas a Kaggle que fallaron en una corrida anterior
dataset <- preprocesar(archivo_dataset)
message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ",
        nrow(dataset), " filas x ", ncol(dataset), " columnas")

# semilla afuera, variante adentro: asi todas las variantes quedan comparables desde la primera semilla
for (semilla in semillas) {
  for (variante in variantes) {
    carpeta <- file.path(carpeta_runs, variante$nombre, paste0("s", semilla))
    if (file.exists(file.path(carpeta, "PARAM.yml"))) {
      message("  ya corrida, salteo: ", carpeta)
      next
    }
    message(format(Sys.time(), "%a %b %d %X %Y"), "  === ", variante$nombre, "  semilla ", semilla,
            "  training desde ", variante$desde, " ===")
    fila <- correr_experimento(
      dataset, variante, semilla, carpeta,
      grid = grid, num_iterations = num_iterations,
      early_stopping_rounds = early_stopping_rounds
    )
    fwrite(fila, file = archivo_resumen, sep = ",", append = file.exists(archivo_resumen))
    message(format(Sys.time(), "%a %b %d %X %Y"), "  fin ", variante$nombre, " s", semilla,
            "  AUC ", round(fila$AUC_valid, 5), "  mejores nl=", fila$num_leaves,
            " mdil=", fila$min_data_in_leaf, " niter=", fila$num_iterations,
            "  (", fila$minutos, " min)")
  }
}

reintentar_pendientes()
message(format(Sys.time(), "%a %b %d %X %Y"), "  terminado en ",
        round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
fread(archivo_resumen)

## Puntajes de Kaggle y test de Wilcoxon
Baja de Kaggle los puntajes de los envíos del experimento y, para cada semilla y variante, calcula la ganancia como el promedio Public de los 6 cortes. Solo entran las corridas que ya tienen los 6 cortes puntuados.

Después compara cada recorte con V0 de a pares por semilla, con el test de Wilcoxon de rangos con signo. La regla quedó fijada antes de ver resultados: un recorte es mejor que V0 si el Wilcoxon da p < 0,05 a su favor. Si gana alguno, la recomendación es entrenar desde su primer mes (si ganan dos, el de mayor diferencia promedio); si no gana ninguno, con todos los meses.

Necesita el `kaggle.json`: lo copia la celda de arranque del z610, o se toma de `/content/buckets/b1/kaggle/` o de `/content/` (subiéndolo con el panel de archivos de la izquierda). Esta celda se puede correr sola, después de la primera celda de código.

In [ ]:
VARIANTES_ORDEN <- c("V0_todo", "V1_sinASPO", "V2_2021", "V3_ultimos4")
PRIMER_MES <- c(V0_todo = 202005, V1_sinASPO = 202011, V2_2021 = 202101, V3_ultimos4 = 202103)

preparar_kaggle()

# la CLI de Kaggle lista hasta 200 envios y no devuelve el token de la pagina siguiente,
# asi que la lista completa se baja con la API de Python, pagina por pagina
writeLines(c(
  "import csv, sys",
  "from kaggle.api.kaggle_api_extended import KaggleApi",
  "from kagglesdk.competitions.types.competition_api_service import ApiListSubmissionsRequest",
  "from kagglesdk.competitions.types.competition_enums import SubmissionGroup, SubmissionSortBy",
  "api = KaggleApi()",
  "api.authenticate()",
  "envios, token = [], ''",
  "for pagina in range(50):",
  "    with api.build_kaggle_client() as kaggle:",
  "        req = ApiListSubmissionsRequest()",
  "        req.competition_name = sys.argv[1]",
  "        req.group = SubmissionGroup.SUBMISSION_GROUP_ALL",
  "        req.sort_by = SubmissionSortBy.SUBMISSION_SORT_BY_DATE",
  "        api._set_paging(req, 200, token, -1)",
  "        resp = kaggle.competitions.competition_api_client.list_submissions(req)",
  "    envios.extend(resp.submissions or [])",
  "    token = getattr(resp, 'next_page_token', '') or ''",
  "    if not token or not resp.submissions:",
  "        break",
  "with open(sys.argv[2], 'w', newline='', encoding='utf-8') as f:",
  "    w = csv.writer(f)",
  "    w.writerow(['ref', 'archivo', 'descripcion', 'fecha', 'estado', 'public'])",
  "    for s in envios:",
  "        w.writerow([s.ref, s.file_name, s.description, s.date, s.status, s.public_score])",
  "print(len(envios), 'envios en la competencia')"
), "bajar_envios_kaggle.py")
invisible(system(paste("python3 bajar_envios_kaggle.py", COMPETENCIA, "envios_kaggle.csv")))

# me quedo con los envios del experimento: el nombre del archivo (o el mensaje) dice variante, semilla y corte
patron_archivo <- "^KA07_(.+)_s([0-9]+)_([0-9]+)\\.csv$"
patron_mensaje <- "^.*exp07 ([^ ]+) s([0-9]+) envios=([0-9]+).*$"
envios_kaggle <- fread("envios_kaggle.csv", colClasses = "character")
envios_kaggle[, public := suppressWarnings(as.numeric(public))]
envios_kaggle[, clave := fifelse(grepl(patron_archivo, archivo), sub(patron_archivo, "\\1|\\2|\\3", archivo),
                         fifelse(grepl(patron_mensaje, descripcion),
                                 sub(patron_mensaje, "\\1|\\2|\\3", descripcion), NA_character_))]
puntajes <- envios_kaggle[!is.na(clave) & grepl("COMPLETE", estado) & !is.na(public)]
puntajes[, c("variante", "semilla", "envios") := tstrsplit(clave, "|", fixed = TRUE)]
puntajes[, `:=`(semilla = as.integer(semilla), envios = as.integer(envios))]
puntajes <- puntajes[variante %in% VARIANTES_ORDEN & envios %in% CORTES]   # solo los 6 cortes
# si un archivo se subio mas de una vez, me quedo con el envio mas reciente
setorder(puntajes, fecha)
puntajes <- puntajes[, .SD[.N], by = .(variante, semilla, envios)]
fwrite(puntajes[, .(variante, semilla, envios, public, fecha, ref, archivo)], "puntajes_kaggle.csv")
message(nrow(puntajes), " envios del experimento con puntaje")

# ganancia Public media por corte y variante
cortes <- puntajes[, .(n = .N, media = mean(public), desvio = sd(public),
                       minimo = min(public), maximo = max(public)), by = .(variante, envios)]
fwrite(cortes, "puntajes_resumen_cortes.csv")
tabla_cortes <- dcast(cortes, envios ~ variante, value.var = "media")
setcolorder(tabla_cortes, c("envios", intersect(VARIANTES_ORDEN, names(tabla_cortes))))
print(tabla_cortes[, lapply(.SD, round, 2), by = envios])

# por corrida: la ganancia es el promedio de los 6 cortes; solo entran las corridas con los 6 puntuados
corridas <- puntajes[, .(cortes = .N, ganancia = mean(public)), by = .(variante, semilla)]
fwrite(corridas, "puntajes_corridas.csv")
completas <- corridas[cortes == length(CORTES)]
if (nrow(corridas) > nrow(completas))
  message("todavia sin todos los cortes (no entran en la comparacion): ",
          corridas[cortes < length(CORTES), paste0(variante, " s", semilla, " (", cortes, " cortes)", collapse = ", ")])

# cada recorte contra V0, de a pares por semilla, con el test de Wilcoxon de rangos con signo
v0 <- completas[variante == "V0_todo", .(semilla, v0 = ganancia)]
pares <- merge(completas[variante != "V0_todo"], v0, by = "semilla")
pares[, dif := ganancia - v0]

resumen <- completas[, .(n_semillas = .N, ganancia_media = mean(ganancia),
                         desvio_entre_semillas = sd(ganancia)),
                     by = variante]
comparacion <- pares[, .(n_pares = .N, dif_media_vs_V0 = mean(dif), semillas_que_superan_V0 = sum(dif > 0),
                         wilcoxon_p = if (.N >= 5) suppressWarnings(wilcox.test(dif, exact = TRUE)$p.value) else NA_real_),
                     by = variante]
resumen <- merge(resumen, comparacion, by = "variante", all.x = TRUE)
resumen <- resumen[order(match(variante, VARIANTES_ORDEN))]

# regla fijada antes de ver resultados: un recorte es mejor que V0 si el Wilcoxon da p < 0,05 a su favor
resumen[variante != "V0_todo", veredicto := fifelse(n_pares < 10, paste0("faltan semillas (", n_pares, " de 10)"),
                                          fifelse(wilcoxon_p < 0.05 & dif_media_vs_V0 > 0, "gana", "no gana"))]
fwrite(resumen, "puntajes_resumen_variantes.csv")
print(resumen[, lapply(.SD, function(x) if (is.numeric(x)) round(x, 4) else x)])

# recomendacion concreta: desde que mes entrenar en la z719
recortes <- resumen[variante != "V0_todo"]
if (nrow(recortes) == 3 && !anyNA(recortes$n_pares) && all(recortes$n_pares >= 10)) {
  ganadoras <- recortes[veredicto == "gana"]
  if (nrow(ganadoras)) {
    mejor <- ganadoras[which.max(dif_media_vs_V0)]
    message("Recomendacion: entrenar desde ", PRIMER_MES[[mejor$variante]], " (", mejor$variante, ")")
  } else {
    message("Recomendacion: ningun recorte cumple la regla, entrenar con todos los meses desde 202005")
  }
} else {
  message("La recomendacion sale cuando las tres comparaciones tengan las 10 semillas")
}